In [ ]:
import os, sys
import pandas as pd
from scipy.stats import ranksums
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import cooler
import anndata
from statsmodels.sandbox.stats.multicomp import multipletests as FDR
from scipy.stats import chi2_contingency
import xarray as xr
from scipy.signal import find_peaks
from scipy.stats import norm
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from PyComplexHeatmap import *
import pybedtools
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))
import warnings
warnings.filterwarnings("ignore")
from PyComplexHeatmap import *
matplotlib.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['figure.dpi'] = 80
matplotlib.rcParams['savefig.dpi']=300

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/HBA_pseudobulk"
indir = None
res=25000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
prefix='Subclass+Region' # used to define indir: {prefix}_compartment
group_col='Subclass' # Region # used to define figdir, and cell_types (group_order), match the col in metadata.columns
parent_col="Class" #used to sort the group_col (MajorType)
sort_cols=None
stratify_col = None #Subclass
stratify_figdir=None
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
split_order = ["Nonneuron", "Glut", "Splatter", "Subpallium GABA"]
figdir=None
cpu = 16
gene_annotation=None
marker_path=None
marker_sheet=None
species='human'
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']
## stratify
# prefix = "Subclass+Region"
# group_col = "Region"
# parent_col = 'Class'
# stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/domain"
outname = "1.Class.diff_boundary.ipynb"


In [ ]:
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if not indir is None:
    indir=os.path.expanduser(indir)
else:
    assert not workdir is None
    indir=os.path.join(workdir,f"{prefix}_domain")
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"domain",group_col)
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)
if not gene_annotation is None:
    gene_annotation=os.path.expanduser(gene_annotation)
print(indir,figdir,group_col)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes[chrom_sizes.index!='chrY']
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

In [ ]:
# chrom_sizes

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)
sorted_groups=metadata[group_col].unique().tolist()
stratify2group={}
group_order=[]
if stratify_col is None:
    for g in sorted_groups:
        sname=g.replace(' ','_')
        if os.path.exists(os.path.join(indir,sname,'25K_domain.boundary.h5ad')):
            group_order.append(g)
else:
    if stratify_figdir is None:
        metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str) #apply(lambda x:x.replace(' ','_')).
    else:
        metadata['GroupID']=metadata[group_col]
    stratify_values=metadata[stratify_col].unique().tolist()
    for s in stratify_values:
        s_name=s.replace(' ','_')
        if stratify_figdir is None:
            for g in sorted_groups:
                g_name=g.replace(' ','_')
                ct=f'{s_name}+{g_name}'
                if os.path.exists(os.path.join(indir,ct,'25K_domain.boundary.h5ad')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{s}+{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'
        else:
            D=metadata.loc[:,[group_col,stratify_col]].drop_duplicates().groupby(stratify_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
            for g in D[s]:
                g_name=g.replace(' ','_')
                if os.path.exists(os.path.join(indir,g_name,'25K_domain.boundary.h5ad')):
                    if s not in stratify2group:
                        stratify2group[s]=[]
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'
group2stratify={}
for k in stratify2group:
    for g in stratify2group[k]:
        group2stratify[g]=k
print(len(group_order),group_order)
print(stratify2group)
print(group2stratify)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
D1=D[group_col].Hex.to_dict()
group_colors={k: D1[k] for k in D1 if k in sorted_groups}
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent=None
print(parent_colors)
print(group_colors)
print(group2parent)

## Domain stats

In [ ]:
if not os.path.exists(os.path.join(figdir, "cell_boundary.h5ad")):
    tad_path_list = [os.path.join(indir,ct.replace(' ','_'),'25K_domain.boundary.h5ad') for ct in group_order] #single cell level domain boundary
    print(len(tad_path_list)) #tad_path_list
    boundary = []
    for xx in tad_path_list:
        boundary.append(anndata.read_h5ad(xx))
    boundary = anndata.AnnData.concatenate(*boundary, index_unique=None)
    # boundary: shape = (No of cells, No of bins), binarized values of boundries
    print(boundary.shape)
    boundary.X = boundary.X.astype(int)
    boundary = boundary[boundary.obs.index.isin(metadata.index)].copy()
    boundary = boundary[:, ~boundary.var['chrom'].isin(['chrX', 'chrY', 'chrM', 'chrL'])].copy()
    boundary.obs = metadata.loc[boundary.obs.index]
    boundary.raw = boundary.copy()
    # boundary.X.toarray() # X is a sparse matrix, values: 0,1,2, what is the meaning of 0-2?, this boundary shared by how many domains?
    # 1 means either start or end of a domain,
    # 2 means both start and end of two domains

    domain_length={}
    cells=boundary.obs.index.tolist()

    # Domain count and size
    def compute_domain_length(i):
        # computate the total length (bp) for all domains in one cell
        global res, boundary
        tmp = boundary.raw.X[i]
        tmp = np.repeat(tmp.indices, tmp.data).reshape((-1, 2))
        return (tmp[:, 1] - tmp[:, 0]).sum() * res

    with ProcessPoolExecutor(cpu) as exe:
        future_dict = {}
        for i, cell in enumerate(cells):
            future = exe.submit(compute_domain_length, i=i)
            future_dict[future] = cell

        for future in as_completed(future_dict):
            cell = future_dict[future]
            # print(f'{cell} finished.')
            domain_length[cell]=future.result()
    boundary.obs['domain_coverage'] = boundary.obs.index.to_series().map(domain_length)
    # total length (bp) for all domains in one cell
    boundary.obs['domain_count'] = boundary.raw.X.sum(axis=1).A1 / 2 #sum by row (cell), A1: convert from (n_cel,1) to (n_cell,)
    boundary.obs['boundary_count'] = boundary.raw.X.getnnz(axis=1) #No of non zeros, sum by row(cell)
    boundary.obs['domain_size'] = boundary.obs['domain_coverage'] / boundary.obs['domain_count']
    boundary.obs['domain_size']=boundary.obs['domain_size'].fillna(0)
    # domain_size is the avg domain length (bp)
    drop_cols=[]
    for col in boundary.obs.columns:
        if boundary.obs[col].nunique()==0:
            drop_cols.append(col)
    boundary.obs.drop(drop_cols,axis=1,inplace=True)
    boundary.strings_to_categoricals()
    boundary.write_h5ad(os.path.join(figdir, "cell_boundary.h5ad"))
else:
    boundary=anndata.read_h5ad(os.path.join(figdir, "cell_boundary.h5ad"))
boundary.obs

In [ ]:
boundary.obs[group_col].value_counts()

In [ ]:
# test whether different cell class have sigficantly different domain_count
import scipy.stats as stats
if not parent_col is None and stratify_col is None:
    for cell_class in boundary.obs[parent_col].unique():
        a=boundary.obs.loc[boundary.obs[parent_col]==cell_class,'domain_count']
        b=boundary.obs.loc[boundary.obs[parent_col]!=cell_class,'domain_count']
        print(cell_class,a.median(),b.median(),ranksums(a, b))
if not stratify_col is None:
    for stratify in stratify2group:
        df1=boundary.obs.loc[boundary.obs[stratify_col]==stratify]
        df1['GroupID']=df1.loc[:,[stratify_col,group_col]].apply(lambda x:'+'.join([str(i) for i in x]),axis=1)
        if len(stratify2group[stratify])<2:
            continue
        values=[df1.loc[df1.GroupID==f"{stratify}+{g}",'domain_count'].values for g in stratify2group[stratify] if f"{stratify}+{g}" in df1.GroupID.tolist()]
        if len(values)<2:
            continue
        f_stat, p_value = stats.f_oneway(*tuple(values))
        print(stratify,f_stat,p_value)


In [ ]:
def plot_domain_size(cell_types,data,group_col,group_colors,parent_col,group2parent,stratify_col,title):
    ncols=len(cell_types)
    # order=cell_types if stratify_col is None else [ct.split('+')[1] for ct in cell_types if ct.split('+')[1] in data[group_col].tolist()]
    order=[col for col in data.groupby(group_col)['domain_size'].mean().sort_values().index.tolist() if col in cell_types]
    fig, axes = plt.subplots(3, 1, figsize=(max(3.5,ncols*0.25), 6), sharex='all',
                             dpi=300,gridspec_kw=dict(hspace=0.15))
    ax = axes[0]
    if not stratify_col is None:
        ax.set_title(title)
    sns.boxplot(data=data, x=group_col, y='domain_count',
                order=order, showfliers=False, palette=group_colors, ax=ax)
    ax.grid(axis='y',which='major',linestyle='--')
    ax.set_ylabel("Domain Count")
    ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)
    ax = axes[1]
    sns.boxplot(data=data, x=group_col, y='domain_coverage',
                order=order, showfliers=False, palette=group_colors, ax=ax)
    ax.grid(axis='y',which='major',linestyle='--')
    ax.set_ylabel("Domain Coverage")
    ax.tick_params(axis="x",which="both",top=False,bottom=False,labeltop=False,labelbottom=False)
    ax = axes[2]
    sns.boxplot(data=data, x=group_col, y='domain_size',
                order=order, showfliers=False, palette=group_colors, ax=ax)
    ax.grid(axis='y',which='major',linestyle='--')
    ax.set_ylabel("Domain Size")
    ax.xaxis.label.set_visible(False)
    ax.set_xticklabels(order, rotation=90)
    if not stratify_col is None or parent_col is None:
        for tick in ax.get_xticklabels():
            tick.set_color(group_colors.get(tick.get_text(),'black'))
    elif not parent_col is None:
            for tick in ax.get_xticklabels():
                try:
                    color=parent_colors[group2parent[tick.get_text()]]
                except:
                    color=group_colors[tick.get_text()]
                tick.set_color(color)
    # plt.subplots_adjust(left=0.2, right=0.95, bottom=0.2, top=0.95)
    plt.tight_layout()
    outname=os.path.join(figdir,f'{group_col}.domain_stat_boxplot.pdf') if stratify_col is None else os.path.join(figdir,f'{title}.domain_stat_boxplot.pdf')
    plt.savefig(outname,transparent=True, bbox_inches='tight',dpi=300)

In [ ]:
if stratify_col is None:
    plot_domain_size(group_order,boundary.obs,group_col,group_colors,parent_col,group2parent,stratify_col,None)
else:
    for stratify in stratify2group:
        cell_types=[f"{stratify}+{g}" for g in stratify2group[stratify]] if stratify_figdir is None else stratify2group[stratify]
        if len(cell_types) < 2:
            continue
        data=boundary.obs.loc[boundary.obs[stratify_col]==stratify]
        plot_domain_size(cell_types,data,group_col,group_colors,parent_col,group2parent,stratify_col,stratify)
    plot_domain_size(boundary.obs[group_col].unique().tolist(),boundary.obs,group_col,group_colors,parent_col,group2parent,None,group_col)
# domain_size = domain_coverage / domain_count

In [ ]:
# if not stratify_col is None:
#     df=boundary.obs.reset_index().loc[:,[stratify_col,group_col,'domain_count','domain_coverage','domain_size']].groupby([stratify_col,group_col]).median().reset_index()
#     # df1=df.pivot(index=stratify,columns=use_group,values='domain_count')
#     # df2=df.pivot(index=stratify,columns=use_group,values='domain_coverage')
#     # df3=df.pivot(index=stratify,columns=use_group,values='domain_size')
#     print(df)
#     if not parent_col is None:
#         df_rows=boundary.obs.reset_index().loc[:,[stratify_col,parent_col]].drop_duplicates().set_index(stratify_col)
#         print(df_rows)
#         # df.domain_size=df.domain_size.apply(lambda x:np.log2(x) if not pd.isna(x) else np.nan)
#         row_ha = HeatmapAnnotation(
#                     parent_col=anno_simple(df_rows[parent_col],colors=parent_colors,#height=5,
#                                          # add_text=True,legend=False,text_kws=dict(fontweight='bold')
#                                         ),
#                     legend=True,wgap=0.5,axis=0,orientation='left',
#                     label_kws=dict(visible=False,color='black',rotation=45,fontsize=8,fontstyle='italic',
#                                       alpha=0.9,verticalalignment='center',horizontalalignment='left'))
#         y_order=df_rows.sort_values(parent_col).index.tolist()
#         row_split=df_rows[parent_col]
#     else:
#         row_ha=None
#         y_order=df[stratify_col].unique().tolist()
#         row_split=None

#     for col in ['domain_size','domain_count','domain_coverage']:
#         plt.figure(figsize=(3.5, 5))
#         vmin=df[col].quantile(0.05)
#         vmax=df[col].quantile(0.95)
#         # vmin=0 if col!='domain_coverage' else 2.65e9
#         cm = DotClustermapPlotter(data=df,
#                                   y=stratify_col,x=group_col,value=col,c=col,#s='domain_size',
#                                   row_cluster=False,col_cluster=False,
#                                   cmap='turbo', #RedYellowBlue
#                                   left_annotation=row_ha,
#                                   y_order=y_order,
#                                   x_order=df[group_col].unique().tolist(),
#                                   marker='o',vmin=vmin,vmax=vmax,
#                                   show_rownames=True,show_colnames=True,row_dendrogram=False,
#                                   col_names_side='bottom',row_names_side='left',
#                                   xticklabels_kws={
#                                       'labelrotation': 90,
#                                       'labelcolor': 'black','labelsize':14
#                                   },
#                                   row_split=row_split,spines=False,
#                                   verbose=1,legend_gap=8,#linewidths=0.5,edgecolors='black',
#                                  )
#         plt.tight_layout()
#         plt.savefig(os.path.join(figdir,f'{col}.domain_stat_dotplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
#         plt.show()

## [Cell type] X [bins] boundary counts

In [ ]:
group_name=group_col if stratify_col is None else 'GroupID'
if not os.path.exists(os.path.join(figdir, 'bound_count.tsv.gz')):
    bound_count_ct = []
    cts = boundary.obs[group_name].unique().tolist()
    print(cts)
    for ct in cts:
        bound_count_ct.append(boundary.raw.X[boundary.obs[group_name] == ct].getnnz(axis=0)) #sum by column (bin)
        # getnnz, number of nonzero terms of a sparse matrix, 0: columns
    bound_count_ct = pd.DataFrame(bound_count_ct, index=cts,
                                  columns=boundary.raw.var['chrom'].astype(str) + '_' + (
                                      boundary.raw.var['start'] // res).astype(str))
    # bound_count_ct: number of boundry (1) for each bin (columns) in each cell type
    # cell_count_ct = boundary.obs[group_name].value_counts().loc[cts]
    # bound_prob_ct = (bound_count_ct / cell_count_ct[:,None]).T
    bound_count_ct=bound_count_ct.T #transform to make it read faster (No. of rows greater than No. of columns)
    bound_count_ct.to_csv(os.path.join(figdir, 'bound_count.tsv.gz'),sep='\t')
else:
    bound_count_ct = pd.read_csv(os.path.join(figdir, 'bound_count.tsv.gz'),sep='\t',index_col=0)
bound_count_ct

In [ ]:
cell_count_ct = boundary.obs[group_name].value_counts().loc[bound_count_ct.columns.tolist()]
cell_count_ct #.to_dict()

## Differential boundary

In [ ]:
bound_prob_ct = bound_count_ct / cell_count_ct
bound_prob_ct

The insulation profile depicts the local frequency of contacts between DNA sequences along the chromosome diagonal of the Hi-C contact matrix.
High contact frequencies within a window suggest a more open and interacting chromatin region.
Conversely, low contact frequencies (dips or minima) in the profile indicate potential boundaries between topologically associating domains (TADs).

Peak Prominence:

Cooltools utilizes peak prominence to quantify the strength of these boundaries based on the dips in the insulation profile. Peak prominence is a mathematical concept that measures the relative height of a peak compared to its surrounding "baseline."

Steps involved:

Cooltools identifies local minima (dips) within the insulation profile. These dips potentially represent boundaries between TADs.
For each dip, cooltools considers a window around it. The window size can be user-defined or based on heuristics within the cooltools implementation.
Within this window, cooltools calculates the difference between the minimum value (the dip) and the average of values on either side of the dip. This difference represents the peak prominence for that specific boundary.

Higher Prominence, Stronger Boundary:

A higher peak prominence value indicates a deeper dip in the insulation profile. This suggests a stronger decrease in contact frequency, potentially signifying a more well-defined boundary between TADs.
Conversely, a lower peak prominence value indicates a shallower dip or less pronounced decrease in contact frequency, suggesting a weaker boundary.


insulation score,The score represents the sum of contact frequencies within the window,

High insulation score: Indicates a low frequency of contacts between upstream and downstream regions within the window. This suggests a stronger boundary between potential topologically associating domains (TADs) within A or B compartments. TADs are regions of chromatin that interact more frequently with each other compared to outside regions.

Low insulation score: Indicates a high frequency of contacts within the window. This suggests a more open and interactive region, potentially within an active A compartment.

Overall, the insulation score profile provides insights into the compartmental organization of the genome and the existence of boundaries between potential TADs.

Valleys (local minima, lower insulations core) = TAD boundaries

Peaks or plateaus (higher instulation score) = internal TAD regions, where local chromatin interactions are dense.

TAD boundaries often act as barriers to enhancer–promoter communication

Genes near strong TAD boundaries are more likely to be insulated from distal regulatory elements in neighboring domains.

Enhancers and promoters within the same TAD are more likely to interact.

Active genes often reside within TADs, particularly near loop anchors or within sub-TADs, where enhancers cluster.

In [ ]:
ins_count = xr.open_dataset(os.path.join(indir,'25K_domain.insulation.nc')) #pseudobulk level insulation score
# ins_count['ratio'] = ins_count['__xarray_dataarray_variable__']
judge = ins_count.get("bin_chrom").isin(['chrX','chrY','chrM','chrL'])
ins_count = ins_count.sel({'bin': ~judge})
print(ins_count)
ins = ins_count['__xarray_dataarray_variable__'].to_pandas().T
ins.columns=[col.replace('_',' ') for col in ins.columns.tolist()]
ins=ins.loc[:,bound_count_ct.columns.tolist()]

df_bin = ins_count[['bin_chrom', 'bin_start', 'bin_end']].to_pandas()
df_bin.columns = df_bin.columns.str.split('_').str[1]
df_bin.index = df_bin['chrom'] + '_' + (df_bin['start'] // res).astype(str)
ins.index=df_bin.index
print(np.nanmin(ins.values),np.nanmax(ins.values))
ins #between 0-1

In [ ]:
# We found increased insulation within neurons relative to glial cells (Figure 1H). ref: PMID: 37811875
# Plot a box plot to show the statistically significant decreases in insulation scores in neurons cell types
if stratify_col is None:
    data=ins.stack().reset_index()
    data.columns=['ID',group_name,"Insulation score"] #group_name=group_col if stratify_col is None else 'GroupID'
    order=data.groupby(group_name)['Insulation score'].median().sort_values().index.tolist()

    plt.figure(figsize=(max(3.5,len(order)*0.25), 2))
    ax=sns.boxplot(data=data, x=group_name, y='Insulation score',order=order,
                showfliers=False, palette=group_colors)
    ax.set_xticklabels(order, rotation=90)
    ax.xaxis.label.set_visible(False)
    ax.grid(which='major',axis='y',linestyle='--')

    if not parent_col is None:
        for tick in ax.get_xticklabels():
            tick.set_color(parent_colors[group2parent[tick.get_text()]])
    plt.tight_layout()
    outname=os.path.join(figdir,f'{group_col}.domain.insulation_score.boxplot.pdf')
    plt.savefig(outname,transparent=True, bbox_inches='tight',dpi=300)

else:
    for stratify in stratify2group:
        cell_types=[f"{stratify}+{g}" for g in stratify2group[stratify]] if stratify_figdir is None else stratify2group[stratify]
        if len(cell_types) < 2:
            continue
        data=ins.loc[:,cell_types].stack().reset_index()
        data.columns=['ID',group_name,"Insulation score"]
        if stratify_figdir is None:
            data[group_name]=data[group_name].apply(lambda x:x.split('+')[1])
        order=data.groupby(group_name)['Insulation score'].median().sort_values().index.tolist()

        plt.figure(figsize=(max(3.5,len(order)*0.25), 5))
        ax=sns.boxplot(data=data, x=group_name, y='Insulation score',order=order,
                    showfliers=False, palette=group_colors)
        ax.set_xticklabels(order, rotation=90)
        ax.grid(which='major',axis='y',linestyle='--')
        ax.xaxis.label.set_visible(False)
        ax.set_title(stratify,fontsize=14)
        for tick in ax.get_xticklabels():
            tick.set_color(group_colors.get(tick.get_text(),'black'))
        plt.tight_layout()
        outname=os.path.join(figdir,f'{stratify}.domain.insulation_score.boxplot.pdf')
        plt.savefig(outname,transparent=True, bbox_inches='tight',dpi=300)

    # merge all cell types
    data=ins.stack().reset_index()
    data.columns=['ID',group_name,"Insulation score"]
    if stratify_figdir is None:
        data[group_name]=data[group_name].apply(lambda x:x.split('+')[1])
    order=data.groupby(group_name)['Insulation score'].median().sort_values().index.tolist()

    plt.figure(figsize=(max(3.5,len(order)*0.25), 5))
    ax=sns.boxplot(data=data, x=group_name, y='Insulation score',order=order,
                showfliers=False, palette=group_colors)
    ax.set_xticklabels(order, rotation=90)
    ax.grid(which='major',axis='y',linestyle='--')
    ax.xaxis.label.set_visible(False)
    ax.set_title(group_col,fontsize=14)
    for tick in ax.get_xticklabels():
        tick.set_color(group_colors.get(tick.get_text(),'black'))
    plt.tight_layout()
    outname=os.path.join(figdir,f'{group_col}.domain.insulation_score.boxplot.pdf')
    plt.savefig(outname,transparent=True, bbox_inches='tight',dpi=300)

In [ ]:
group_black_list_path = os.path.join(workdir, f"{prefix}_raw/loop.blacklist.10kb.bed") #10kb blacklist
bkl = pd.read_csv(group_black_list_path, sep='\t', header=None, index_col=None)
df_bin['bklfilter'] = True
for c in chrom_sizes.index:
    chrfilter = (df_bin['chrom']==c)
    tmp = df_bin.loc[chrfilter.values]
    tmp.iloc[:10, -1] = False
    tmp.iloc[-10:, -1] = False
    for xx,yy in (bkl.loc[bkl[0]==c, [1,2]] // res).drop_duplicates().values:
        tmp.iloc[max([0,xx-2]):(yy+2), -1] = False # removed the near 3 bins around the 10kb blacklist
    df_bin.loc[chrfilter] = tmp.copy()

print(df_bin['bklfilter'].sum())
df_bin=df_bin.loc[bound_count_ct.index.tolist()]
df_bin

In [ ]:
def diff_bound(bound_count_ct, cell_count_ct):
    tmp = cell_count_ct - bound_count_ct #total number of cells for each MajorType - number of boundary cells
    stats = np.zeros(bound_count_ct.shape[0]) # No. of 25kb bins, rows are bins, columns are cell types
    pv = np.ones(bound_count_ct.shape[0])
    binfilter = np.logical_and(bound_count_ct.sum(axis=1) > 0, tmp.sum(axis=1) > 0) # sum by row (bins), keep the bins with at least one diff boundary
    for i in range(bound_count_ct.shape[0]):
        if binfilter[i]:
            contig = [bound_count_ct[i,:], tmp[i,:]]
            stats[i], pv[i], _, _ = chi2_contingency(contig) #test different boundary using chi2 square test
    fdr = FDR(pv, 0.01, 'fdr_bh')[1]
    return stats, fdr

# def shuffle_ct(i):
#     global cell_count_ct, sc_border, leg
#     np.random.seed(i)
#     label = np.random.permutation(sc_border.obs[f'{ct_key}'])
#     bound_count_ct = np.array([sc_border.X[label == xx].getnnz(axis=0) for xx in leg])
#     bound_prob_ct = bound_count_ct / cell_count_ct[:, None]
#     return diff_bound(bound_count_ct, cell_count_ct)[0]

def diff_bound_bulk(ins_count):
    stats = np.zeros(ins_count.shape[2])
    pv = np.ones(ins_count.shape[2])
    binfilter = (ins_count.min(axis=(0, 1)) > 0)
    for i in range(ins_count.shape[2]):
        if binfilter[i]:
            stats[i], pv[i], _, _ = chi2_contingency(ins_count[:, :, i])
    fdr = FDR(pv, 0.01, 'fdr_bh')[1]
    return stats, pv

In [ ]:
chi2_stat, chi2_fdr = diff_bound(bound_count_ct.values, cell_count_ct.values)
ave = np.mean(chi2_stat[chi2_stat>0])
stdev = np.std(chi2_stat[chi2_stat>0])
df_bin['chi2filter'] = (((chi2_stat - ave) / stdev)>norm.isf(0.025)) # =norm.ppf(0.975) when x = certain value, the area on the left side is 0.975
# norm.isf(0.025) = 1.95,norm.sf(1.95)=0.025, area of > 1.95, right part in norm curve
df_bin

In [ ]:
# boundary strength (peak prominence)
df_bin['ins_peak'] = 0 # old name: ins_lm
for ct in group_order:
    sel = []
    prominence=[]
    for c in chrom_sizes.index:
        idx = np.where(df_bin['chrom']==c)[0] #row index
        if len(idx)>0:
            #find bins with low insulation score for each cell type
            peaks, properties = find_peaks(-ins[ct].iloc[idx], distance=5,prominence=0,width=5) #prominence=0.2, width=5 (added myself, no such para in Jingtian's docu)
            sel.append(idx.min() + peaks)
            prominence.append(properties['prominences'])
    sel = np.concatenate(sel)
    df_bin.loc[df_bin.index[sel], 'ins_peak'] = 1 # in at least one cell type
    df_bin.loc[df_bin.index[sel], 'boundary_strength'] = np.concatenate(prominence) # in at least one cell type
print(df_bin.ins_peak.sum())
df_bin['probdiff'] = (bound_prob_ct.max(axis=1) - bound_prob_ct.min(axis=1)).values # delta boundary probability
df_bin['chi2_stat'] = chi2_stat.copy() #chi2 squere test statistics
df_bin['chi2_fdr'] = chi2_fdr.copy()
df_bin['insfc'] = ((ins.max(axis=1)+0.01) / (ins.min(axis=1)+0.01)).values # insulation score fold change
df_bin

In [ ]:
assert bound_prob_ct.index.tolist() == df_bin.index.tolist()

In [ ]:
cols=bound_prob_ct.columns.tolist() #bound_prob_ct
df_bin['Group']=df_bin.index.to_series().map(bound_prob_ct.apply(lambda x:cols[np.argmax(x)], axis=1).to_dict())

In [ ]:
sel = []
thres = np.min(chi2_stat[chi2_fdr<1e-3])
for c in chrom_sizes.index:
    idx = np.where(df_bin['chrom']==c)[0]
    if len(idx)>0:
        data = chi2_stat[idx]
        peaks, _ = find_peaks(data, height=thres, distance=5)
        sel.append(idx.min() + peaks)
        
sel = np.concatenate(sel)
df_bin['chi2_stat_peak'] = 0 #old name: diff_sc
df_bin.loc[df_bin.index[sel], 'chi2_stat_peak'] = 1 # peak identified using chi2 square statistics
df_bin.loc[:, df_bin.dtypes=='category'] = df_bin.loc[:, df_bin.dtypes=='category'].astype(str)
df_bin.to_csv(os.path.join(figdir,'bin_stats.tsv.gz'),sep='\t')
df_bin

## Plot diff boundary

In [ ]:
# df_bin=pd.read_csv(os.path.join(figdir,'bin_stats.tsv.gz'),sep='\t',index_col=0)
# df_bin

In [ ]:
ax=sns.displot(df_bin.boundary_strength,bins=100)
ax2 = ax.ax.twinx()
ax=sns.kdeplot(df_bin.boundary_strength,ax=ax2,color='red')

In [ ]:
f1=(df_bin['chi2filter'] & df_bin['chi2_stat_peak'] & df_bin['bklfilter'] & df_bin['ins_peak'] & (df_bin['probdiff']>0.05) 
       & (df_bin.boundary_strength > 0.05)
      )
f1.sum()

In [ ]:
f2=(df_bin['chi2filter'] & df_bin['chi2_stat_peak'] & df_bin['bklfilter'] & df_bin['ins_peak'] & (df_bin['probdiff']>0.05) & (df_bin['insfc']>1.5) &(df_bin.boundary_strength > 0.05))
f2.sum()

In [ ]:
df_bin['position']=df_bin.chrom+':'+df_bin.start.map(str)+':'+df_bin.end.map(str)
df_bin.loc[f1].sort_values('probdiff',ascending=False)

In [ ]:
df_cols=ins.columns.to_frame(name=group_name)
if not stratify_col is None:
    if stratify_figdir is None:
        df_cols[stratify_col]=df_cols[group_name].apply(lambda x:x.split('+')[0])
        df_cols[group_col]=df_cols[group_name].apply(lambda x:x.split('+')[1])
    else:
        df_cols[stratify_col]=df_cols[group_name].map(group2stratify)
        df_cols[group_col]=df_cols[group_name]
if not parent_col is None:
    df_cols[parent_col]=df_cols[group_name].map(group2parent)
    df_cols.sort_values([parent_col,group_name],inplace=True)
else:
    if split_order is None:
        df_cols.sort_values(group_name,inplace=True)
    else:
        df_cols=df_cols.loc[split_order]
df_cols.head()

In [ ]:
df_rows=df_bin.loc[f1]
if not stratify_col is None:
    if stratify_figdir is None:
        df_rows[stratify_col]=df_rows.Group.apply(lambda x:x.split('+')[0])
        df_rows[group_col]=df_rows.Group.apply(lambda x:x.split('+')[1])
    else:
        df_rows[stratify_col]=df_rows.Group.map(group2stratify)
        df_rows[group_col]=df_rows.Group
if not parent_col is None:
    df_rows[parent_col]=df_rows.Group.map(group2parent)
if not parent_col is None and not stratify_col is None:
    parent2stratify=df_rows.loc[:,[parent_col,stratify_col]].drop_duplicates().groupby(parent_col)[stratify_col].apply(lambda x:x.unique().tolist()).to_dict()
    row_order=[]
    for parent in parent2stratify:
        for stratify in parent2stratify[parent]:
            for g in stratify2group[stratify]:
                if stratify_figdir is None:
                    rows=df_rows.loc[df_rows['Group']==f"{stratify}+{g}"].sort_values('probdiff',ascending=False).index.tolist()
                else:
                    rows=df_rows.loc[df_rows['Group']==g].sort_values('probdiff',ascending=False).index.tolist()
                row_order.extend(rows)
elif not stratify_col is None:
    row_order=[]
    for stratify in stratify2group:
        for g in stratify2group[stratify]:
            if stratify_figdir is None:
                rows=df_rows.loc[df_rows['Group']==f"{stratify}+{g}"].sort_values('probdiff',ascending=False).index.tolist()
            else:
                rows=df_rows.loc[df_rows['Group']==g].sort_values('probdiff',ascending=False).index.tolist()
            row_order.extend(rows)
else:
    row_order=[]
    for g in df_cols[group_name].tolist():
        rows=df_rows.loc[df_rows.Group==g].sort_values('probdiff',ascending=False).index.tolist()
        row_order.extend(rows)
df_rows=df_rows.loc[row_order]
df_cols=df_cols.loc[df_rows.Group.unique().tolist()]
df_rows.head()

In [ ]:
from pym3c.utils import read_markers
df_marker=read_markers(
        marker_path,marker_sheet,
        species=species,marker_cols = ['RNA_markers', 'Methylation_markers','SelectedMarkers']
    )
marker_genes=list(set(df_marker.Markers.sum()))
df_marker.head()

In [ ]:
if not gene_annotation is None:
    df1=df_rows.iloc[:,:3]
    df1['ID']=df1.index.tolist()
    # extend 1Mb
    # df1.start=df1.start.apply(lambda x:x-50000 if x>=50000 else x)
    # df1.end=df1.end+50000
    bed1 = pybedtools.BedTool.from_dataframe(df1)
    df2=pd.read_csv(os.path.expanduser(gene_annotation),sep='\t',header=None,names=['chr','start','end','gene_id','gene_symbol','strand','gene_type'])
    bed2 = pybedtools.BedTool.from_dataframe(df2)
    intersection = bed1.intersect(bed2, wa=True, wb=True,loj=True) #f=0.7,F=0.7,r=True
    df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2']+df2.columns.tolist()[3:])
    # read dmg table
    # df_dmg=pd.read_csv(os.path.expanduser(dmg),sep='\t')
    # df_dmg=df_dmg.loc[(df_dmg.wilcox_statistic<0) & (df_dmg.Group==group_col)]
    # gene_list=df_dmg.Gene.unique().tolist()
    df=df.loc[df.gene_type=='protein_coding']
    if not marker_path is None:
        df=df.loc[df.gene_symbol.isin(marker_genes)]
    df['length_intersection'] = df[['end', 'end2']].min(axis=1) - df[['start', 'start2']].max(axis=1)
    df['fraction1']=df['length_intersection'] / (df.end-df.start)
    df['fraction2']=df['length_intersection'] / (df.end2-df.start2)
    df['fraction']=df.loc[:,['fraction1','fraction2']].min(axis=1)
    df.sort_values('fraction',ascending=False,inplace=True)
    df=df.loc[df.fraction >= 0.1 ]
    df.drop_duplicates('ID',keep='first',inplace=True)
    
    D=df.groupby('ID').gene_symbol.apply(lambda x:','.join(x.unique().tolist()) if x.nunique() > 0 else '').to_dict()
    df_rows['Gene']=df_rows.index.to_series().map(D)
    print(df_rows.Gene.nunique())
df_rows

In [ ]:
col_ha_dict={
    }
if parent_col in df_cols.columns.tolist():
    tmp_colors={}
    for parent in df_cols[parent_col].unique():
        tmp_colors[parent]=parent_colors[parent]
    col_ha_dict[parent_col]=anno_simple(
        df_cols[parent_col],#height=4,
         colors=tmp_colors,
         add_text=False,text_kws={'fontweight':'bold','color':'black'},legend=True)
    col_split=df_cols[parent_col]
    col_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]
else:
    col_split=None
    col_split_order=None
if stratify_col in df_cols.columns.tolist():
    tmp_colors={}
    for stratify in df_cols[stratify_col].unique():
        tmp_colors[stratify]=stratify_colors[stratify]
    col_ha_dict[stratify_col]=anno_simple(
        df_cols[stratify_col],height=4,
         colors=tmp_colors,
         add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True)
    if col_split is None:
        col_split=df_cols[stratify_col]
        col_split_order=df_cols[stratify_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[stratify_col].tolist()]

    if group_col in df_cols.columns.tolist():
        tmp_colors={}
        for g in df_cols[group_col].unique():
            tmp_colors[g]=group_colors[g]
        col_ha_dict[group_col]=anno_simple(
            df_cols[group_col],height=4,
             colors=tmp_colors,
             add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True)
        show_colnames=True
else:
    show_colnames=True

if len(col_ha_dict)>0:
    col_ha1 = HeatmapAnnotation(**col_ha_dict,axis=1,
             verbose=0,label_kws=dict(visible=False)
                               )
    col_ha2 = HeatmapAnnotation(**col_ha_dict,axis=1,
             verbose=0,label_kws=dict(visible=True)
                               )
else:
    col_ha1=None
    col_ha2=None

if parent_col in df_rows.columns.tolist():
    row_split=df_rows[parent_col]
    row_split_order=col_split_order
else:
    row_split=None
    row_split_order=None

if not gene_annotation is None:
    right_ha=HeatmapAnnotation(
        Gene=anno_label(df_rows.Gene,merge=True,extend=True,colors='black',fontsize=6,relpos=(0,0.5)),
        axis=0,verbose=0,orientation='right')
else:
    right_ha=None
    
data=bound_prob_ct.loc[df_rows.index.tolist(),df_cols.index.tolist()]
data_ins=ins.loc[df_rows.index.tolist(),df_cols.index.tolist()]

cm1 = ClusterMapPlotter(
            data=data, top_annotation=col_ha1, #left_annotation=None,right_annotation=right_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,
            row_cluster_method='ward',row_cluster_metric='cosine',
            col_split=col_split, col_split_order=col_split_order,
            # xticklabels_kws={'labelrotation':90,'labelcolor':'black','labelsize':10},
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
            col_split_gap=0.4,cmap='BlueDarkRed18',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
            row_split=row_split,row_split_order=row_split_order,
            rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
            plot=False,label='normalized\nprobability',legend_gap=14,
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1), #z_score=0,
)
cm2 = ClusterMapPlotter(
            data=data_ins, top_annotation=col_ha2, right_annotation=right_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
            col_split_gap=0.4,cmap='Blues_r',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
            row_split=row_split,row_split_order=row_split_order,
            rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
            plot=False,label='Insulation score',legend_gap=14,
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1), #z_score=0,
)

cmlist=[cm1,cm2]

In [ ]:
from PyComplexHeatmap.utils import despine
nrows,ncols=data.shape
width=max(ncols*0.4,3.5)
plt.figure(figsize=(width,10))
ax,legend_axes=composite(cmlist=cmlist, main=0,col_gap=15,
                         legend_hpad=25, legend_gap=10)
cm1.ax.set_title("Boundary Prob",fontsize=14)
cm2.ax.set_title("Insulation Score",fontsize=14)
ax.set_ylabel(str(data.shape[0])+" Diff Boundaries",color='black',fontsize=14)
ax.set_axis_on()
despine(ax=ax, left=True, bottom=True,top=True,right=True)
ax.tick_params(axis="both",which="both",left=False,
                right=False,labelleft=False,labelright=False,
                top=False,bottom=False,labeltop=False,labelbottom=False)
plt.savefig(os.path.join(figdir,f'{group_col}.diff_boundry_prob.heatmap.pdf'),transparent=True, bbox_inches='tight',dpi=300)

In [ ]:
df_rows.to_csv(os.path.join(figdir,"diff_boundary.tsv"),sep='\t')
# df_rows.loc[df_rows.Gene.notna()].sort_values('probdiff',ascending=False)

In [ ]:
if not stratify_col is None:
    x_order=[]
    for parent in split_order:
        x_order.extend(df_rows.loc[df_rows[parent_col]==parent][stratify_col].unique().tolist())
    print(x_order)
    hue_order=df_rows[group_col].value_counts().index.tolist()[::-1]
    plt.figure(figsize=(10,3))
    ax=sns.countplot(data=df_rows,x=group_col,palette=group_colors,order=hue_order)
    ax.set_xticklabels(hue_order, rotation=90)
    ax.set_title("No. of differential boundary",fontsize=16)
    plt.savefig(os.path.join(figdir,f'Number_of_diff_boundry.{group_col}.barplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

    df=df_rows.Group.value_counts().to_frame()
    if stratify_figdir is None:
        df[stratify_col]=df.index.to_series().apply(lambda x:x.split('+')[0])
        df[group_col]=df.index.to_series().apply(lambda x:x.split('+')[1])
    else:
        df[stratify_col]=df.index.to_series().map(group2stratify)
        df[group_col]=df.index
    df=df.set_index([stratify_col,group_col]).unstack().fillna(0)
    df=df.apply(lambda x:x/x.sum(),axis=1)['count']

    plt.figure
    ax=df.loc[x_order].plot(kind='bar',stacked=True,color=group_colors,align='center', width=0.9)
    ax.legend(loc='upper left',frameon=True,bbox_to_anchor=(1.0,1),ncol=2,labelspacing=0.15,handleheight=0.5)
    ax.set_ylim(0,1)
    sns.despine(ax=ax,top=True,right=True)
    plt.savefig(os.path.join(figdir,f'percentage_of_diff_boundry.{group_col}.barplot.pdf'),transparent=True, bbox_inches='tight',dpi=300)
    plt.show()

## Write bigwig

In [ ]:
df_bin

In [ ]:
for i,col in enumerate(['chrom','start','end']):
    D=df_bin[col].to_dict()
    bound_prob_ct.insert(i,col,bound_prob_ct.index.to_series().map(D))
    ins.insert(i,col,ins.index.to_series().map(D))
bound_prob_ct

In [ ]:
ins

In [ ]:
diff_boundary=df_rows.loc[:,['chrom','start','end','probdiff','insfc','Group']]
diff_boundary.sort_values('probdiff',ascending=False)

In [ ]:
def df2bw(data,chrom_size_path,outfile,col,chrom="chrom",start='start',end='end'):
    import cooler,pyBigWig
    chrom_size_path = os.path.expanduser(chrom_size_path)
    chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
    chrom_sizes = chrom_sizes[chrom_sizes.index != 'chrY']
    df_chr = chrom_sizes.to_frame()
    df_chr['N'] = df_chr.index.to_series().apply(
        lambda x: int(x.lstrip('chr')) if x.lstrip('chr') not in ['X', 'Y'] else 30)  # sort by chrom int (1,2,3..)
    df_chr.sort_values('N', inplace=True)
    chrom_sizes = chrom_sizes.loc[df_chr.index.tolist()]
    chrom_sizes_list = [(k, v) for k, v in chrom_sizes.to_dict().items()]
    
    data.sort_values([chrom,start,end],ascending=True,inplace=True)
    f = pyBigWig.open(outfile, "w")
    f.addHeader(chrom_sizes_list)
    for c in chrom_sizes.index: #order of chrom should be the same as chrom_sizes_list
        tmp1=data.loc[data[chrom] == c]
        if tmp1.shape[0]==0:
            continue
        f.addEntries(tmp1[chrom].tolist(), tmp1[start].tolist(),ends=tmp1[end].tolist(),values=tmp1[col].tolist(),validate=True)
    f.close()

In [ ]:
# boundary probability
outdir=os.path.join(figdir,"boundary_probability")
if not os.path.exists(outdir):
    os.mkdir(outdir)
for col in bound_prob_ct.columns.tolist()[3:]:
    print(col,end=',')
    outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bw")
    df2bw(bound_prob_ct,chrom_size_path,outfile,col,chrom="chrom",start='start',end='end')

In [ ]:
# insulation scores
outdir=os.path.join(figdir,"insulation_scores")
if not os.path.exists(outdir):
    os.mkdir(outdir)
for col in ins.columns.tolist()[3:]:
    print(col,end=',')
    outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bw")
    df2bw(ins,chrom_size_path,outfile,col,chrom="chrom",start='start',end='end')

In [ ]:
# diff_boundary
outdir=os.path.join(figdir,"diff_boundary")
if not os.path.exists(outdir):
    os.mkdir(outdir)
for col in diff_boundary.Group.unique():
    print(col,end=',')
    outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bw")
    df=diff_boundary.loc[diff_boundary.Group==col]
    df2bw(df,chrom_size_path,outfile,col='probdiff',chrom="chrom",start='start',end='end')

In [ ]:
diff_boundary.sort_values('probdiff',ascending=False)

## write domain (TAD)

In [ ]:
boundary=anndata.read_h5ad(os.path.join(indir,'25K_domain.boundary.h5ad'))
boundary.var['ID']=boundary.var.apply(lambda x:'-'.join([str(i) for i in x]),axis=1)
boundary_df=boundary.to_df()
boundary_df.columns=boundary.var['ID'].tolist()
use_rows=[ct.replace(' ','_') for ct in group_order]
boundary_df=boundary_df.loc[use_rows].T
boundary_df

In [ ]:
# convert boundary to domain
R=[]
for col in boundary_df.columns.tolist(): # cell type level domain doundaries, not on single cell level
    df=boundary_df.loc[boundary_df[col]>0,col]
    # 1 means either start or end of a domain,
    # 2 means both start and end of two domains
    bounds=df.index.tolist()
    values=df.tolist()
    i=0
    while i <= len(bounds)-2:
        chrom1,start1,end1=bounds[i].split('-')
        chrom2,start2,end2=bounds[i+1].split('-')
        if chrom1!=chrom2:
            continue
        start=(int(start1)+int(end1)) / 2
        end=(int(start2)+int(end2)) / 2
        R.append([chrom1,int(start),int(end),col])
        boundary_n=values[i+1]
        if boundary_n==2:
            i+=1
        else:
            i+=2
df=pd.DataFrame(R,columns=['chrom','start','end','Group'])
df

In [ ]:
# TAD
outdir=os.path.join(figdir,"domain")
if not os.path.exists(outdir):
    os.mkdir(outdir)
for col in df.Group.unique():
    print(col,end=',')
    outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bed")
    df1=df.loc[df.Group==col,['chrom','start','end']]
    df1.to_csv(outfile,sep='\t',index=False,header=False)

In [ ]:
# domain boundary
outdir=os.path.join(figdir,"boundary")
if not os.path.exists(outdir):
    os.mkdir(outdir)
celltypes=boundary_df.columns.tolist()
boundary_df.insert(0,'chrom',boundary_df.index.to_series().apply(lambda x:x.split('-')[0]))
boundary_df.insert(1,'start',boundary_df.index.to_series().apply(lambda x:int(x.split('-')[1])))
boundary_df.insert(2,'end',boundary_df.index.to_series().apply(lambda x:int(x.split('-')[2])))
for ct in celltypes:
    print(ct,end='r')
    s=boundary_df.loc[:,['chrom','start','end',ct]]
    s=s.loc[s[ct] > 0]
    s.sort_values(['chrom','start'],inplace=True)
    outfile=os.path.join(outdir,f"{ct.replace(' ','_')}.bed")
    s.loc[:,['chrom','start','end']].to_csv(outfile,sep='\t',index=False,header=False)